# ACSIncome Business Rule Gap — V4

Phiên bản này tách **controlled benchmark** khỏi **raw-variable candidate discovery**. Sáu biến dẫn xuất chỉ dùng làm bộ kiểm thử có kiểm soát; các dependency trên biến ACS gốc chỉ được xem là ứng viên cần chuyên gia và codebook thẩm định.

In [ ]:
# 0. Chuẩn bị môi trường — chạy được cả local và Google Colab
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = 'https://github.com/HaThienNgan/18_HaThienNgan_-AMH.git'
if not Path('src/acs_rule_gap').exists():
    target = Path('/content/18_HaThienNgan_-AMH')
    if not target.exists():
        subprocess.run(['git', 'clone', '-q', REPOSITORY, str(target)], check=True)
    os.chdir(target)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Môi trường đã sẵn sàng:', Path.cwd())

In [ ]:
# 1. Imports và cấu hình
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from folktables import ACSDataSource, ACSIncome

from acs_rule_gap.data import create_derived_variables, stratified_sample, stratified_train_test_split
from acs_rule_gap.drift import clean_dds_threshold, leave_one_group_out_dds, paired_dependency_drift
from acs_rule_gap.metrics import pairwise_dependency_scores
from acs_rule_gap.stability import permutation_test_edges, stable_dependency_candidates

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
STATES = ['CA', 'TX', 'NY', 'FL', 'IL']
ANALYSIS_SAMPLE_SIZE = 100_000
QUICK_MODE = True  # đổi thành False khi chạy kết quả chính thức
STABILITY_RESAMPLES = 10 if QUICK_MODE else 50
EDGE_PERMUTATIONS = 49 if QUICK_MODE else 499
DDS_BOOTSTRAPS = 20 if QUICK_MODE else 200
LOO_PERMUTATIONS = 19 if QUICK_MODE else 199
print({'quick_mode': QUICK_MODE, 'stability_resamples': STABILITY_RESAMPLES})

## 2. Tải dữ liệu, lấy mẫu và chia train/test

Discovery chỉ sử dụng train. Test được giữ lại cho generalization, stress test và drift.

In [ ]:
def load_acs_income(states, survey_year='2018'):
    source = ACSDataSource(survey_year=survey_year, horizon='1-Year', survey='person')
    parts = []
    for state in states:
        acs = source.get_data(states=[state], download=True)
        features, labels, _ = ACSIncome.df_to_pandas(acs)
        part = features.copy()
        part['income_label'] = labels.astype(int).to_numpy()
        part['state'] = state
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

raw_df = load_acs_income(STATES)
sampled_df = stratified_sample(raw_df, ANALYSIS_SAMPLE_SIZE, ['state', 'income_label'], RANDOM_SEED)
df = create_derived_variables(sampled_df)
train_df, test_df = stratified_train_test_split(df, ['state', 'income_label'], 0.20, RANDOM_SEED)

profile = pd.DataFrame([
    {'dataset': 'raw', 'rows': len(raw_df), 'columns': raw_df.shape[1]},
    {'dataset': 'analysis', 'rows': len(df), 'columns': df.shape[1]},
    {'dataset': 'train', 'rows': len(train_df), 'columns': train_df.shape[1]},
    {'dataset': 'test', 'rows': len(test_df), 'columns': test_df.shape[1]},
])
display(profile)
display(pd.crosstab(df['state'], df['income_label'], normalize='index').round(3))

## 3. Controlled benchmark và kiểm tra negative control

Các cạnh dưới đây được tạo bởi người nghiên cứu. Kết quả recovery chỉ đánh giá pipeline, không chứng minh đây là rule ACS chính thức. Negative control hoán vị biến đích; nếu metric vẫn tìm lại rule, thiết kế đang có vấn đề.

In [ ]:
benchmark_catalog = pd.DataFrame([
    {'lhs': 'AGEP', 'rhs': 'age_group'},
    {'lhs': 'SCHL', 'rhs': 'education_group'},
    {'lhs': 'WKHP', 'rhs': 'working_hours_group'},
    {'lhs': 'WKHP', 'rhs': 'fulltime_status'},
    {'lhs': 'MAR', 'rhs': 'marital_group'},
    {'lhs': 'income_label', 'rhs': 'income_group'},
])
BENCHMARK_COLS = [
    'AGEP', 'SCHL', 'MAR', 'WKHP', 'income_label',
    'age_group', 'education_group', 'working_hours_group',
    'fulltime_status', 'marital_group', 'income_group',
]
benchmark_edges = set(map(tuple, benchmark_catalog[['lhs', 'rhs']].to_numpy()))

def recovery_table(data, label):
    scores = pairwise_dependency_scores(data, BENCHMARK_COLS)
    rows = []
    for metric in ['ExactFD', 'QStrength', 'TheilsU']:
        ranked = scores.sort_values([metric, 'avg_lhs_support'], ascending=False)
        for k in [6, 12, 20]:
            top = ranked.head(k)
            hits = sum((r.lhs, r.rhs) in benchmark_edges for r in top.itertuples())
            rows.append({
                'dataset': label, 'metric': metric, 'K': k, 'hits': hits,
                'Precision@K': hits / k, 'Recall@K': hits / len(benchmark_edges),
            })
    return scores, pd.DataFrame(rows)

benchmark_scores, benchmark_recovery = recovery_table(train_df, 'controlled_benchmark')
negative = train_df.copy()
rng = np.random.default_rng(RANDOM_SEED)
for rhs in benchmark_catalog['rhs']:
    negative[rhs] = rng.permutation(negative[rhs].to_numpy())
_, negative_recovery = recovery_table(negative, 'permuted_negative_control')
recovery = pd.concat([benchmark_recovery, negative_recovery], ignore_index=True)
display(recovery.round(3))

pivot = recovery[recovery['K'] == 12].pivot(index='metric', columns='dataset', values='Recall@K')
pivot.plot(kind='bar', figsize=(9, 4), ylim=(0, 1.05), title='Recovery so với negative control — K=12')
plt.ylabel('Recall@K')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Dependency ứng viên trên biến ACS gốc

Bootstrap stability selection giảm phụ thuộc vào một lần lấy mẫu. Cardinality/support guard hạn chế quan hệ giả do LHS gần duy nhất. Permutation test và Benjamini–Hochberg kiểm soát bằng chứng thống kê trên shortlist.

In [ ]:
RAW_COLS = ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'RAC1P', 'income_label']
candidates = stable_dependency_candidates(
    train_df, RAW_COLS,
    resamples=STABILITY_RESAMPLES, sample_size=20_000, top_k=20,
    max_rhs_cardinality=30, min_avg_lhs_support=20, seed=RANDOM_SEED,
)
shortlist = candidates[candidates['selection_frequency'] >= 0.80].head(10)
if shortlist.empty:
    shortlist = candidates.head(10)

permutation = permutation_test_edges(
    train_df, shortlist, permutations=EDGE_PERMUTATIONS, sample_size=20_000, seed=RANDOM_SEED
)
test_scores = pairwise_dependency_scores(test_df, RAW_COLS)[['lhs', 'rhs', 'TheilsU']].rename(columns={'TheilsU': 'TheilsU_test'})
candidate_report = (
    shortlist.merge(permutation, on=['lhs', 'rhs'], how='left')
             .merge(test_scores, on=['lhs', 'rhs'], how='left')
)
candidate_report['generalization_gap'] = candidate_report['TheilsU_median'] - candidate_report['TheilsU_test']
candidate_report['status'] = np.where(
    (candidate_report['selection_frequency'] >= 0.80)
    & (candidate_report['fdr_q_value'] < 0.05)
    & (candidate_report['generalization_gap'].abs() <= 0.05),
    'candidate_for_expert_review',
    'retain_for_exploration',
)
display(candidate_report.round(4))

## 5. Paired Dependency Drift Score

Clean và noisy dùng cùng vị trí dòng. DDS chỉ tính ngoài đường chéo, vì đường chéo luôn bằng 1 và không chứa thông tin drift.

In [ ]:
DERIVED_DOMAINS = {
    'age_group': ['Under18', 'WorkingAge', 'Senior'],
    'education_group': ['BelowHS', 'HS_College', 'BachelorPlus'],
    'working_hours_group': ['VeryLow', 'PartTime', 'Standard', 'Overtime'],
    'fulltime_status': ['Fulltime', 'NonFulltime'],
    'marital_group': ['Married', 'NotMarried'],
    'income_group': ['HighIncome', 'LowIncome'],
}

def inject_derived_errors(clean, rate, seed):
    noisy = clean.copy(deep=True)
    rng = np.random.default_rng(seed)
    selected = rng.choice(len(noisy), size=max(1, round(len(noisy) * rate)), replace=False)
    columns = list(DERIVED_DOMAINS)
    for position in selected:
        column = rng.choice(columns)
        current = noisy.iloc[position][column]
        choices = [value for value in DERIVED_DOMAINS[column] if value != current]
        noisy.iat[position, noisy.columns.get_loc(column)] = rng.choice(choices)
    return noisy

DDS_COLS = BENCHMARK_COLS
clean_threshold, clean_null = clean_dds_threshold(
    test_df, DDS_COLS, sample_size=3_000, repetitions=DDS_BOOTSTRAPS, seed=RANDOM_SEED
)
dds_rows = []
for rate in [0.01, 0.05, 0.10]:
    for seed in [42, 52, 62, 72, 82]:
        noisy = inject_derived_errors(test_df, rate, seed)
        dds_rows.append({
            'noise_rate': rate, 'seed': seed,
            'DDS': paired_dependency_drift(test_df, noisy, DDS_COLS, sample_size=3_000, seed=seed),
        })
dds = pd.DataFrame(dds_rows)
dds_summary = dds.groupby('noise_rate', as_index=False).agg(DDS_mean=('DDS', 'mean'), DDS_std=('DDS', 'std'))
display(dds_summary.round(6))
plt.errorbar(dds_summary['noise_rate'], dds_summary['DDS_mean'], yerr=dds_summary['DDS_std'], marker='o')
plt.axhline(clean_threshold, linestyle='--', label=f'Clean 95% = {clean_threshold:.4f}')
plt.xlabel('Tỷ lệ dòng bị thay đổi')
plt.ylabel('Paired DDS')
plt.title('Dependency drift khi phá vỡ controlled rules')
plt.legend()
plt.tight_layout()
plt.show()

## 6. State leave-one-out không tạo drift giả

Hàm tự loại `state` khỏi ma trận rồi tạo null distribution bằng permutation. Khác biệt theo bang được diễn giải là heterogeneity theo bối cảnh, không mặc định là lỗi.

In [ ]:
state_drift = leave_one_group_out_dds(
    test_df, group_col='state', columns=RAW_COLS + ['state'],
    sample_size=2_000, permutation_reps=LOO_PERMUTATIONS, seed=RANDOM_SEED,
)
display(state_drift.round(6))
plt.figure(figsize=(8, 4))
plt.bar(state_drift['state'], state_drift['DDS_vs_other_groups'])
plt.scatter(state_drift['state'], state_drift['null_DDS_95'], marker='_', s=500, label='Permutation null 95%')
plt.ylabel('DDS so với các bang còn lại')
plt.title('State-level dependency heterogeneity')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Cách kết luận

- Recovery tốt ở controlled benchmark chứng minh pipeline hoạt động trong điều kiện có kiểm soát.
- Negative control phải làm recovery suy giảm.
- Dependency raw vượt stability, permutation/FDR và generalization chỉ là ứng viên cho chuyên gia.
- DDS cảnh báo thay đổi cấu trúc; row-level validator mới xác định dòng vi phạm.
- Khác biệt theo bang có thể là heterogeneity hợp lệ và cần phân tích bối cảnh.